# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs
1. Deux tests statistiques SciPy avec H0/H1.
2. Segmentation K-Means par agrégation Spark, log1p et StandardScaler.
3. Dashboard Streamlit alimenté uniquement par Gold.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName("J4_Stats_Segmentation_Dashboard").getOrCreate()
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"


## 1. Inférence Statistique
### Exercice 1.1 — Formuler H0/H1, projeter les colonnes utiles et conclure.

**Test 1 — Mann–Whitney :** H0 : les distributions des montants des commandes livrées en retard et à l'heure sont identiques ; H1 : elles diffèrent. Test non paramétrique pour deux groupes indépendants.

**Test 2 — Spearman :** H0 : la corrélation de rang entre frais de transport et montant de commande est nulle ; H1 : elle est non nulle. Test adapté aux distributions asymétriques.

Seuil de décision : α = 0,05. Une p-value inférieure à 0,05 indique le rejet de H0 ; elle ne démontre pas de causalité.


In [ ]:
# Sélection et échantillonnage borné : pas de toPandas() sur la table Silver entière.
df_base = spark.read.format("delta").load(SILVER_MAIN_PATH)
df_sample = (df_base.select("order_value", "freight_value", "is_late")
             .where("order_value IS NOT NULL AND freight_value IS NOT NULL AND is_late IS NOT NULL")
             .sample(False, 0.30, seed=42).limit(20000).toPandas())
df_sample["order_value"] = pd.to_numeric(df_sample["order_value"], errors="coerce")
df_sample["freight_value"] = pd.to_numeric(df_sample["freight_value"], errors="coerce")
df_sample = df_sample.dropna()
groupe_a = df_sample.loc[df_sample["is_late"] == True, "order_value"]
groupe_b = df_sample.loc[df_sample["is_late"] == False, "order_value"]
assert len(groupe_a) > 0 and len(groupe_b) > 0, "Deux groupes non vides sont nécessaires."
stat_1, p_val_1 = stats.mannwhitneyu(groupe_a, groupe_b, alternative="two-sided")
rho, p_val_2 = stats.spearmanr(df_sample["freight_value"], df_sample["order_value"])
print(f"Mann-Whitney U={stat_1:.3f} p={p_val_1:.6g} :", "rejet H0" if p_val_1 < .05 else "H0 non rejetée")
print(f"Spearman rho={rho:.4f} p={p_val_2:.6g} :", "rejet H0" if p_val_2 < .05 else "H0 non rejetée")
print("Interprétation : association statistique, pas causalité. Échantillon borné à 20 000 observations.")


## 2. Segmentation clients avec K-Means

Les variables agrégées par client sont le nombre de commandes, le montant total, le panier médian, le taux de retard et la récence. Les variables positives sont transformées par log1p ; les variables sont standardisées avant K-Means.


In [ ]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/customer_segments"
# Une ligne par client : agrégation distribuée côté Spark.
from pyspark.sql import Window
base = spark.read.format("delta").load(SILVER_MAIN_PATH)
max_date = base.agg(F.max("purchase_date")).first()[0]
assert max_date is not None, "Aucune date de commande disponible."
features = (base.where(F.col("customer_unique_id").isNotNull() & F.col("order_value").isNotNull())
 .groupBy("customer_unique_id")
 .agg(F.countDistinct("order_id").alias("frequency"),
      F.sum("order_value").alias("total_spend"),
      F.expr("percentile_approx(order_value, 0.5)").alias("median_basket"),
      F.avg(F.when(F.col("is_late") == True, 1.0).otherwise(0.0)).alias("late_rate"),
      F.max("purchase_date").alias("last_purchase"))
 .withColumn("recency_days", F.datediff(F.lit(max_date), F.col("last_purchase")))
 .drop("last_purchase"))
# Limite explicite pour protéger le driver ; pour très grands datasets, utiliser Spark MLlib.
entities_pd = features.orderBy("customer_unique_id").limit(120000).toPandas()
assert len(entities_pd) >= 3, "Au moins 3 clients requis pour K-Means."
cols = ["frequency", "total_spend", "median_basket", "late_rate", "recency_days"]
for c in cols:
    entities_pd[c] = pd.to_numeric(entities_pd[c], errors="coerce").fillna(0).clip(lower=0)
X = entities_pd[cols].copy()
for c in ["frequency", "total_spend", "median_basket", "recency_days"]:
    X[c] = np.log1p(X[c])
scaled = StandardScaler().fit_transform(X)
k = min(4, len(entities_pd))
model = KMeans(n_clusters=k, random_state=42, n_init=10)
entities_pd["cluster_id"] = model.fit_predict(scaled).astype(int)
profiles = entities_pd.groupby("cluster_id")[cols].median()
print("Médianes par cluster :")
print(profiles)
# Attribution déterministe de libellés métiers selon le profil médian.
spend_rank = profiles["total_spend"].rank(method="first")
labels = {}
for cluster in profiles.index:
    if profiles.loc[cluster, "late_rate"] >= max(0.25, profiles["late_rate"].quantile(.75)):
        labels[int(cluster)] = "Clients à risque de retard"
    elif spend_rank.loc[cluster] == spend_rank.max():
        labels[int(cluster)] = "Clients à forte valeur"
    elif profiles.loc[cluster, "frequency"] >= profiles["frequency"].median() and profiles.loc[cluster, "recency_days"] <= profiles["recency_days"].median():
        labels[int(cluster)] = "Clients actifs"
    else:
        labels[int(cluster)] = "Clients occasionnels"
entities_pd["segment"] = entities_pd["cluster_id"].map(labels)
# Évite la conversion pandas->Spark ambiguë avec types numpy.
entities_pd["cluster_id"] = entities_pd["cluster_id"].astype("int64")
result = entities_pd[["customer_unique_id", *cols, "cluster_id", "segment"]].copy()
for c in cols: result[c] = result[c].astype(float)
spark.createDataFrame(result).coalesce(2).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_SEGMENTS_PATH)
print(f"Segments enregistrés : {GOLD_SEGMENTS_PATH} ({len(result):,} clients)")


## 3. Livrable Final : Dashboard Streamlit

Le fichier `app/dashboard.py` fourni dans le ZIP ne lit que les tables Delta Gold et propose filtres, visualisations Plotly et explications SQL.


In [ ]:
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")
assert fs.exists(gold_root), "Dossier Gold absent : exécutez J3 puis J4."
gold_tables = [s.getPath().getName() for s in fs.listStatus(gold_root)
               if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))]
assert len(gold_tables) >= 4, "Exécutez J3 (3 tables) puis J4 (segments)."
print("=== Tables Gold pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")
